# Compute a transfer function

Atmospheric emission can dominate the noise budget and must be removed from the time-ordered data before mapping. This filtering inevitably projects out correlated sky modes, suppressing signal at large angular scales. The spatial transfer function $T(u)$ provides a direct estimate of this scale-dependent signal loss. `maria` includes a built-in function to compute this, using the cross-spectrum between the output and the known input map:

$$T(u) = \frac{\mathrm{Re}\langle \tilde{s}_{\rm in}^*(u)\,\tilde{s}_{\rm out}(u)\rangle}{\langle|\tilde{s}_{\rm in}(u)|^2\rangle}$$

Please note that the cross-spectrum is used instead of the power ratio $P_{\rm out}/P_{\rm in}$ because noise in the output is uncorrelated with the input and averages to zero, leaving an unbiased estimate.

## Input sky

In this tutorial, we simulate a two-frequency observation of a galaxy cluster at 150 and 270 GHz and recover a map with the `BinMapper`. We load the same map at both frequencies — any difference in the recovered signal will come from the instrument and the filtering alone.

In [ ]:
import maria
import matplotlib.pyplot as plt
import numpy as np
from maria.io import fetch

map_filename = fetch("maps/cluster2.fits")

input_map = maria.map.load(filename=map_filename, center=(0, -23))
input_map.data *= 1e4
input_map.plot(slices="all", cmap="RdYlBu_r")

## Instrument and observation plan

We use TolTEC on the LMT, keeping only the 150 and 270 GHz arrays and dropping the 220 GHz one for efficiency. Because the beam is diffraction-limited, the 270 GHz array resolves angular scales roughly 1.8× finer than the 150 GHz array.

In [ ]:
import maria
from maria.instrument import Band

f090 = Band(center=150e9, width=20e9, NET_RJ=100e-6)
f150 = Band(center=270e9, width=40e9, NET_RJ=100e-6)

array = {"field_of_view": 0.25, 
         "beam_spacing": 1.8,
         "primary_size": 12, 
         "shape": "circle",
         "bands": [f090, f150]}

instrument = maria.get_instrument(array=array)

print(instrument)
instrument.plot()

In [ ]:
from maria import Planner

planner = Planner(
    target=input_map,
    start_time="2024-01-01T22:00:00",
    site="llano_de_chajnantor",
    constraints={"el": (60, 90)},
)

plans = planner.generate_plans(
    total_duration=1800,
    sample_rate=50,
    scan_type="daisy",
)

plans[0].plot()
plans

## Simulation

Passing `map=input_map` attaches the ground-truth sky to each output TOD so the mapper can propagate it to the output map automatically.

In [ ]:
sim = maria.Simulation(
    instrument,
    plans=plans,
    site="llano_de_chajnantor",
    map=input_map,
    atmosphere="2d",
    atmosphere_kwargs={"weather": {"pwv": 0.5}},
    # cmb="generate",
    # cmb_kwargs={"nside": 256},
)

print(sim)

tods = sim.run()
tods[0].plot()

## Mapmaking

Common-mode subtraction (`remove_modes`) and per-detector spline removal (`remove_spline`) suppress large-scale correlated signal. Both operations remove power at long time scales, which maps to large angular scales on the sky, i.e., where $T$ will fall below 1.

> **Note:** `map_postprocessing` is intentionally left empty here. Any map-level post-processing (smoothing, filtering, etc.) applied after binning would itself suppress or modify signal in the output map, and its effect would be absorbed into the measured $T(u)$. To isolate the transfer function of the TOD filtering alone, always set `map_postprocessing={}` when computing transfer functions.

In [ ]:
from maria.mapping import MaximumLikelihoodMapper

ml_mapper = MaximumLikelihoodMapper(
    tods=tods,
    units="K_RJ",
    stokes="I",
    resolution=0.5 / 60,
    tod_preprocessing={
        "remove_spline": {"knot_spacing": 60, "remove_el_gradient_order": 1},
    },
    map_postprocessing={},
)


In [ ]:
ml_mapper.binner.tods[0].plot()

In [ ]:
ml_mapper.map.plot(slices="all")

In [ ]:
ml_mapper.fit(epochs=2, 
              max_steps_per_epoch=50, 
              plot=True,
              plot_kwargs={"slices": "all"})

output_map = ml_mapper.map

## Transfer function

To compute the transfer function, call `transfer_function()` on the output map. This will return a `TransferFunction` object containing the transfer function $T(u)$ and the corresponding spatial frequencies $u$. The transfer function can be plotted to visualize the scale-dependent signal loss.

In [ ]:
tf = output_map.transfer_function(window=True, input_map=input_map)
print(tf)

To visualize the transfer function, it is possible to use the built-in `plot()` method. Solid curves show the measured $T(u)$, while dashed curves show the Gaussian beam per channel. The large-scale rolloff reflects the TOD filtering above, while the small-scale rolloff tracks the beam.

In [ ]:
tf.plot(x_unit="arcmin")

Finally, individual channels can be selected with `slices=dict(nu=[...])`, consistent with how `slices` is used in `.plot()`:

In [ ]:
tf.plot(slices=dict(nu=[0]), x_unit="arcmin")

## Apodization window

Before the FFT, `transfer_function()` multiplies both maps by a 2D apodization window to suppress spectral leakage from sharp map edges. The `window` argument controls which window is applied:

| Value | Shape | When to use |
|-------|-------|-------------|
| `"hann"` (default) | Full cosine taper to zero at both edges | Strongest leakage suppression; best when the field is much larger than the scales of interest, since it discards edge signal |
| `"tukey"` | Cosine taper on outer `taper` fraction, central region = 1 | Mild apodization that preserves most of the map signal and substantially reduces edge leakage |
| `np.ndarray` of shape `(ny, nx)` | User-supplied 2D window | Full control — e.g. to mask point sources or weight by the hit-count map |
| `False` / `None` | No window (rectangular) | Maximum large-scale sensitivity; avoid unless the map has periodic or artificially zero-padded boundaries |

The `taper` parameter (default 0.1) sets the fraction of each axis tapered by the Tukey roll-off. Increasing it toward 1 makes the Tukey window approach a Hann window.

> **Tip:** for typical BinMapper output maps, where valid signal extends close to the boundary, `"tukey"` with a small `taper` is preferred over `"hann"`. The full Hann taper downweights map edges heavily, effectively shrinking the usable field and worsening large-scale mode recovery.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)

windows = [
    ("tukey", dict(window="tukey", taper=0.1)),
    ("hann", dict(window="hann")),
    ("none", dict(window=False)),
]

for i, (label, kwargs) in enumerate(windows):
    tf_w = output_map.transfer_function(slices=dict(nu=[0]), input_map=input_map, **kwargs)
    tf_w.plot(ax=ax, x_unit="arcmin", add_beam=False)
    ax.lines[-1].set_label(label)
    ax.lines[-1].set_color(f"C{i}")

ax.legend(frameon=False, fontsize=9)
plt.show()